# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Growing pages are longer and younger than declining pages
- **Where does the label come from?** The paper defines trend direction from a 30d vs previous-30d impressions change threshold, so any "growing vs declining" label depends on that constructed rule.
- **What does the validation design test?** This table is an observational cohort comparison, so it tests association in this snapshot, not whether page length/age changes would cause growth.
- **Would this survive grouped/time validation?** Could this gap shrink if compared within client/topic and publication period (grouped/time-aware split) rather than pooled across all pages?

### Finding 2: 365+ day refreshed pages show 3.2x health and 57x impressions
- **Where does the label come from?** The freshness result is compared against trend/health outcomes that are computed from rolling windows, so label construction and window alignment matter directly.
- **What does the validation design test?** The comparison is descriptive for selected refreshed vs non-refreshed groups; selection into refresh could explain part of the lift.
- **Would this survive grouped/time validation?** Would a client-held-out or forward-time evaluation with matched untreated pages keep a similar effect size, or is some lift explained by who got chosen for refresh?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

feature_path_candidates = [
    Path("../../data/processed/refresh_feature_vector.csv"),
    Path("data/processed/refresh_feature_vector.csv"),
    Path(
        "/home/runner/work/flyrank-ml-internship/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"
    ),
]
feature_path = next((p for p in feature_path_candidates if p.exists()), None)
if feature_path is None:
    raise FileNotFoundError(
        "refresh_feature_vector.csv not found in expected locations"
    )

df = pd.read_csv(feature_path)

numeric_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_cols = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

X_full = pd.get_dummies(
    df[numeric_cols + categorical_cols], columns=categorical_cols
)
y_full = df["is_declining_label"]


def precision_at_k(y_true, score, k):
    k = min(k, len(y_true))
    top_k_idx = np.argsort(score)[::-1][:k]
    return y_true.values[top_k_idx].mean()


def evaluate_split(train_idx, test_idx):
    X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]
    df_test = df.iloc[test_idx].copy()

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test), columns=X_test.columns, index=X_test.index
    )

    visible = df_test["impressions_90d"] >= 500
    slip = (df_test["avg_position"] > 20) & (df_test["avg_position"] > 0)
    stale = df_test["days_since_last_update"] >= 90
    baseline_score_test = visible * (
        slip * df_test["impressions_90d"]
        + stale * df_test["impressions_90d"] * 0.3
    )

    log_reg = LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)
    log_reg.fit(X_train_scaled, y_train)

    forest = RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    )
    forest.fit(X_train, y_train)

    scores = {
        "Baseline": baseline_score_test.values,
        "LogisticRegression": log_reg.predict_proba(X_test_scaled)[:, 1],
        "RandomForest": forest.predict_proba(X_test)[:, 1],
    }

    rows = []
    base_rate = y_test.mean()
    for name, score in scores.items():
        row = {
            "method": name,
            "precision@20": precision_at_k(y_test, score, 20),
            "precision@50": precision_at_k(y_test, score, 50),
            "precision@100": precision_at_k(y_test, score, 100),
            "average_precision": average_precision_score(y_test, score),
            "roc_auc": roc_auc_score(y_test, score),
        }
        if name != "Baseline":
            preds = (score >= 0.5).astype(int)
            row["precision"] = precision_score(y_test, preds)
            row["recall"] = recall_score(y_test, preds)
            row["f1"] = f1_score(y_test, preds)
            row["accuracy"] = accuracy_score(y_test, preds)
        rows.append(row)

    table = pd.DataFrame(rows).set_index("method")
    table["base_rate"] = base_rate
    return table


# Before: plain random split (no grouping)
all_idx = np.arange(len(df))
train_idx_random, test_idx_random = train_test_split(
    all_idx, test_size=0.2, random_state=RANDOM_STATE
)
random_results = evaluate_split(train_idx_random, test_idx_random)

# After: grouped split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx_grouped, test_idx_grouped = next(
    gss.split(X_full, y_full, groups=df["client_id"])
)

assert (
    set(df.iloc[train_idx_grouped]["client_id"])
    & set(df.iloc[test_idx_grouped]["client_id"])
    == set()
), "client leakage in grouped split!"
grouped_results = evaluate_split(train_idx_grouped, test_idx_grouped)

comparison = pd.concat(
    {
        "Random split (before)": random_results.round(3),
        "Grouped split (after)": grouped_results.round(3),
    },
    axis=1,
)

comparison

Random split (before)                             \
                            precision@20 precision@50 precision@100   
method                                                                
Baseline                            0.60         0.58          0.59   
LogisticRegression                  0.90         0.86          0.85   
RandomForest                        0.95         0.96          0.96   

                                                                               \
                   average_precision roc_auc precision recall     f1 accuracy   
method                                                                          
Baseline                       0.558   0.533       NaN    NaN    NaN      NaN   
LogisticRegression             0.734   0.717     0.665  0.758  0.709    0.660   
RandomForest                   0.792   0.775     0.708  0.774  0.739    0.703   

                             Grouped split (after)                             \
                   base_rate          precision@20 precision@50 precision@100   
method                                                                          
Baseline               0.545                  0.30         0.38          0.36   
LogisticRegression     0.545                  0.70         0.72          0.71   
RandomForest           0.545                  0.75         0.66          0.63   

                                                                               \
                   average_precision roc_auc precision recall     f1 accuracy   
method                                                                          
Baseline                       0.502   0.492       NaN    NaN    NaN      NaN   
LogisticRegression             0.604   0.616     0.576  0.721  0.641    0.587   
RandomForest                   0.602   0.608     0.582  0.647  0.613    0.582   

                              
                   base_rate  
method                        
Baseline               0.511  
LogisticRegression     0.511  
RandomForest           0.511

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
from IPython.display import Markdown, display

feature_cols = numeric_cols + categorical_cols
all_columns = set(df.columns)
feature_set = set(feature_cols)

label_source_columns = {"trend_direction", "trend_pct", "is_declining_label"}
label_columns_in_features = sorted(
    feature_set.intersection(label_source_columns)
)

product_flag_keywords = [
    "healthy",
    "fix",
    "zombie",
    "priority",
    "workflow",
    "triage",
    "flag",
    "score",
]
product_like_features = sorted(
    col
    for col in feature_cols
    if any(key in col.lower() for key in product_flag_keywords)
)

window_risk_features = sorted(
    col
    for col in feature_cols
    if ("90d" in col.lower())
    or (col in {"ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"})
)

print("Feature count:", len(feature_cols))
print(
    "Label-source columns present in features:", label_columns_in_features
)
print("Product/priority-like features present:", product_like_features)
print("Potential overlap-risk feature windows (name-based):")
print(window_risk_features)

assert (
    "trend_direction" not in feature_set
), "trend_direction leaked into features"

window_note = """
### Window-overlap note (from the data contract)
- The label `is_declining_label` is derived from `trend_direction`, which uses **impressions_last_30d vs impressions_prev_30d**.
- This Week-5 feature list uses several **90-day aggregates/derived rates** (`*_90d`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`).
- From the static snapshot contract alone, I cannot prove strict non-overlap between every feature window and the label window boundary.
- So the honest audit statement is: direct label columns are excluded, but window overlap risk cannot be fully ruled out without a rebuild that enforces strictly pre-label feature windows.
"""

display(Markdown(window_note))

Feature count: 26
Label-source columns present in features: []
Product/priority-like features present: []
Potential overlap-risk feature windows (name-based):
['ai_traffic_pct', 'ctr', 'engagement_rate', 'log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d', 'scroll_rate']



### Window-overlap note (from the data contract)
- The label `is_declining_label` is derived from `trend_direction`, which uses **impressions_last_30d vs impressions_prev_30d**.
- This Week-5 feature list uses several **90-day aggregates/derived rates** (`*_90d`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`).
- From the static snapshot contract alone, I cannot prove strict non-overlap between every feature window and the label window boundary.
- So the honest audit statement is: direct label columns are excluded, but window overlap risk cannot be fully ruled out without a rebuild that enforces strictly pre-label feature windows.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [3]:
from IPython.display import Markdown, display

original_claim = "For main model, Random forest as a stronger variant of classifcation model as it captures non-linear relationships."

rf_p50 = grouped_results.loc["RandomForest", "precision@50"]
lr_p50 = grouped_results.loc["LogisticRegression", "precision@50"]
rf_ap = grouped_results.loc["RandomForest", "average_precision"]
lr_ap = grouped_results.loc["LogisticRegression", "average_precision"]

rewrite = f"In this dataset and grouped client-holdout split, RandomForest ranked declining pages at precision@50={rf_p50:.3f} (LogisticRegression={lr_p50:.3f}) and average_precision={rf_ap:.3f} (LogisticRegression={lr_ap:.3f}); this is an observed out-of-sample ranking result for decision support, not a causal claim."
why_overreach = "The original sentence asserts model strength from method choice alone; the safer claim ties confidence to measured holdout metrics and keeps it observational."

md = f"""
### Quoted original sentence
> {original_claim}

### Claim-ladder rewrite
{rewrite}

### Why the original overreached
{why_overreach}
"""

display(Markdown(md))


### Quoted original sentence
> For main model, Random forest as a stronger variant of classifcation model as it captures non-linear relationships.

### Claim-ladder rewrite
In this dataset and grouped client-holdout split, RandomForest ranked declining pages at precision@50=0.660 (LogisticRegression=0.720) and average_precision=0.602 (LogisticRegression=0.604); this is an observed out-of-sample ranking result for decision support, not a causal claim.

### Why the original overreached
The original sentence asserts model strength from method choice alone; the safer claim ties confidence to measured holdout metrics and keeps it observational.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.